# 台股基本面選股器
## Quality + Value 投資策略

**策略說明：**  
尋找「體質好且被低估」的股票

**篩選條件：**
- **品質**：ROE > 產業中位數、負債比率 < 60%、營收成長 > 0%
- **估值**：本益比 < 產業平均 × 0.85（有15%折價空間）
- **訊號**：AI自動評分，標註改善幅度大的公司

---

## 1. 環境設定

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 自定義模組
from calculate_metrics import calculate_all_metrics
from fetch_stock_price import add_stock_metrics
from screening import screen_stocks, get_screening_summary

# 顯示設定
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft YaHei']  # 支援中文
plt.rcParams['axes.unicode_minus'] = False

print("✓ 環境設定完成")

## 2. 載入財報資料

從parse_mops_reports.py產生的CSV檔案載入

In [ ]:
# 載入財報資料
financial_data = pd.read_csv('../data/processed/financial_metrics_2025Q3.csv')

print(f"載入 {len(financial_data)} 筆財報資料")
print(f"涵蓋 {financial_data['company_id'].nunique()} 家公司")
print(f"資料品質: {(financial_data['quality'] == 'complete').sum()} 筆完整")

# 只保留完整資料
financial_data = financial_data[financial_data['quality'] == 'complete'].copy()

# 顯示範例
financial_data.head()

## 3. 計算ROE

ROE = 淨利 / 平均股東權益 × 100%

In [ ]:
# 計算ROE
data_with_roe = calculate_all_metrics(financial_data)

# 檢查結果
print("\nROE分布:")
print(data_with_roe['ROE'].describe())

# 顯示ROE範例
data_with_roe[data_with_roe['ROE'].notna()][[
    'company_id', 'company_name', 'net_income_ytd', 
    'shareholders_equity_q', 'ROE'
]].head(10)

## 4. 加入產業分類

**請先完成產業對照表：`../data/industry_mapping.csv`**

格式：
```
company_id,industry
2330,半導體業
2454,半導體業
...
```

In [ ]:
# 載入產業分類
try:
    industry_mapping = pd.read_csv('../data/industry_mapping.csv')
    data_with_industry = data_with_roe.merge(
        industry_mapping, 
        on='company_id', 
        how='left'
    )
    
    print(f"✓ 產業分類載入成功")
    print(f"  涵蓋 {data_with_industry['industry'].notna().sum()} 家公司")
    print(f"  共 {data_with_industry['industry'].nunique()} 個產業")
    
    # 產業分布
    print("\n產業分布（前10）:")
    print(data_with_industry['industry'].value_counts().head(10))
    
except FileNotFoundError:
    print("⚠️ 找不到產業分類檔案，請建立 ../data/industry_mapping.csv")
    print("  暫時使用'未分類'作為產業")
    data_with_industry = data_with_roe.copy()
    data_with_industry['industry'] = '未分類'

## 5. 抓取股價並計算本益比

**注意：** 這步驟會呼叫Yahoo Finance API，需要一些時間（約5-10分鐘）

In [ ]:
# 抓取股價（可選：先用小樣本測試）
# test_sample = data_with_industry.head(50)  # 先測試50家
# complete_data = add_stock_metrics(test_sample, verbose=True)

# 全部資料（需要較長時間）
complete_data = add_stock_metrics(data_with_industry, verbose=True)

# 檢查結果
print("\n本益比分布:")
print(complete_data['PE_ratio'].describe())

# 顯示範例
complete_data[complete_data['PE_ratio'].notna()][[
    'company_id', 'company_name', 'stock_price', 
    'eps_ytd', 'PE_ratio'
]].head(10)

## 6. 執行選股篩選

套用 Quality + Value 策略

In [ ]:
# 執行選股
screened_stocks = screen_stocks(complete_data, industry_col='industry')

# 摘要統計
summary = get_screening_summary(screened_stocks)
print("\n篩選結果摘要:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 7. 結果展示

In [ ]:
# 顯示欄位
display_cols = [
    'company_id', 'company_name', 'industry', 'signal_strength',
    'ROE', 'PE_ratio', 'revenue_yoy', 'net_income_yoy',
    'gross_margin_q', 'debt_ratio_q'
]

# 所有篩選結果
print("\n=== 完整篩選結果 ===")
print(screened_stocks[display_cols].to_string())

In [ ]:
# 強力訊號的公司
strong_signals = screened_stocks[screened_stocks['signal_strength'] == '⭐⭐⭐']

print(f"\n=== ⭐⭐⭐ 強力訊號 ({len(strong_signals)} 家) ===")
print(strong_signals[display_cols].to_string())

## 8. 視覺化分析

In [ ]:
# ROE vs 本益比散點圖
fig, ax = plt.subplots(figsize=(12, 8))

# 不同訊號強度用不同顏色
colors = {'⭐⭐⭐': 'green', '⭐⭐': 'orange', '無': 'gray'}
for signal, color in colors.items():
    data = screened_stocks[screened_stocks['signal_strength'] == signal]
    ax.scatter(
        data['ROE'], 
        data['PE_ratio'], 
        c=color, 
        label=signal,
        alpha=0.6,
        s=100
    )

ax.set_xlabel('ROE (%)', fontsize=12)
ax.set_ylabel('本益比', fontsize=12)
ax.set_title('ROE vs 本益比分布', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 產業分布
if 'industry' in screened_stocks.columns:
    industry_counts = screened_stocks['industry'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    industry_counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('公司數量', fontsize=12)
    ax.set_ylabel('產業', fontsize=12)
    ax.set_title('篩選結果產業分布（前10）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. 匯出結果

In [ ]:
# 匯出篩選結果
output_path = '../data/processed/screening_results.csv'
screened_stocks.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✓ 結果已匯出至: {output_path}")
print(f"  共 {len(screened_stocks)} 家公司")

## 10. 個股分析範例（可選）

深入分析特定公司

In [ ]:
# 選擇一家公司分析（例如台積電）
target_id = '2330'  # 修改這裡來分析不同公司

company_data = screened_stocks[screened_stocks['company_id'] == target_id]

if len(company_data) > 0:
    company = company_data.iloc[0]
    
    print(f"\n=== {company['company_name']} ({company['company_id']}) ===")
    print(f"產業: {company['industry']}")
    print(f"訊號強度: {company['signal_strength']}")
    print(f"\n財務指標:")
    print(f"  ROE: {company['ROE']:.2f}% (產業中位數: {company['industry_median_roe']:.2f}%)")
    print(f"  本益比: {company['PE_ratio']:.2f} (產業平均: {company['industry_avg_pe']:.2f})")
    print(f"  營收成長: {company['revenue_yoy']:.2f}%")
    print(f"  淨利成長: {company['net_income_yoy']:.2f}%")
    print(f"  毛利率: {company['gross_margin_q']:.2f}%")
    print(f"  負債比率: {company['debt_ratio_q']:.2f}%")
    
    # TODO: 這裡可以加入Claude API分析
    # ai_analysis = generate_ai_analysis(company)
    # print(f"\nAI分析:\n{ai_analysis}")
    
else:
    print(f"⚠️ 公司 {target_id} 未通過篩選")

---

## 下一步

1. ✅ 完成基本選股流程
2. 💡 加入AI分析（Claude API）
3. 💡 建立每週更新機制
4. 💡 追蹤歷史績效

---

**投資提醒：**  
本工具僅供研究學習，不構成投資建議。投資有風險，請審慎評估。